In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install xgboost

In [ ]:
# Basic libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix

# Model persistence
import pickle
import json

# Settings
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

print(" All libraries imported successfully!")

In [ ]:
# Load data
df = pd.read_csv('/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(" Data loaded successfully!")
print("Dataset shape:", df.shape)
df.head()

In [ ]:
# Handle missing values
print("Missing values before cleaning:")
print(df.isnull().sum())

# Convert TotalCharges to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Fill missing with median
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# Verify
print("\n Total missing values after cleaning:", df.isnull().sum().sum())

In [ ]:
# Categorical columns list
categorical_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 
                    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                    'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 
                    'PaperlessBilling', 'PaymentMethod']

# Create dummy variables
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Encode target
df_encoded['Churn'] = df_encoded['Churn'].map({'Yes': 1, 'No': 0})

print(f"Original columns: {df.shape[1]}")
print(f"After encoding: {df_encoded.shape[1]}")
print("Encoding complete!")

In [ ]:
# Separate features and target
X = df_encoded.drop(['customerID', 'Churn'], axis=1)
y = df_encoded['Churn']

print(f"Features shape (X): {X.shape}")
print(f"Target shape (y): {y.shape}")
print(f"\nChurn percentage: {y.mean()*100:.2f}%")

In [ ]:
# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")
print(f"\nChurn in training: {y_train.mean()*100:.2f}%")
print(f"Churn in test: {y_test.mean()*100:.2f}%")

In [ ]:
# Baseline Random Forest
rf_baseline = RandomForestClassifier(n_estimators=100, random_state=42)
rf_baseline.fit(X_train, y_train)

# Predict and evaluate
y_pred_baseline = rf_baseline.predict(X_test)
baseline_accuracy = accuracy_score(y_test, y_pred_baseline)

print(f"Baseline Random Forest Accuracy: {baseline_accuracy:.4f} ({baseline_accuracy*100:.2f}%)")
print("\nBaseline Classification Report:")
print(classification_report(y_test, y_pred_baseline))

In [ ]:
# Perform 5-fold cross-validation
rf_cv = RandomForestClassifier(n_estimators=100, random_state=42)
cv_scores = cross_val_score(rf_cv, X_train, y_train, cv=5, scoring='accuracy')

print(" Cross-Validation Results:")
print(f"Scores for each fold: {cv_scores}")
print(f"Mean CV Score: {cv_scores.mean():.4f}")
print(f"Standard Deviation: {cv_scores.std():.4f}")

# Visualize CV scores
plt.figure(figsize=(10,6))
plt.plot(range(1,6), cv_scores, marker='o', linestyle='-', linewidth=2, markersize=8)
plt.axhline(y=cv_scores.mean(), color='r', linestyle='--', label=f"Mean: {cv_scores.mean():.4f}")
plt.xlabel('Fold Number')
plt.ylabel('Accuracy')
plt.title('5-Fold Cross-Validation Scores')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Test different metrics
metrics = ['accuracy', 'precision', 'recall', 'f1']
results = {}

for metric in metrics:
    scores = cross_val_score(rf_cv, X_train, y_train, cv=5, scoring=metric)
    results[metric] = {'mean': scores.mean(), 'std': scores.std()}
    print(f"{metric.capitalize()}: {scores.mean():.4f} (+/- {scores.std():.4f})")

# Create comparison dataframe
cv_comparison = pd.DataFrame(results).T
print("\n Cross-Validation Metrics Summary:")
print(cv_comparison)

In [ ]:
# Parameter grid for Random Forest
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

total_combinations = 3 * 3 * 3 * 3
print(f"Total parameter combinations: {total_combinations}")
print(f"With 5-fold CV: {total_combinations * 5} model trainings!")
print(" This will take a few minutes. Be patient...")

In [ ]:
# Create GridSearchCV object
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

# Fit grid search
print(" Starting Grid Search...")
import time
start_time = time.time()
grid_search.fit(X_train, y_train)
end_time = time.time()

print(f"\n Grid Search completed in {(end_time - start_time)/60:.2f} minutes")

In [ ]:
# Best parameters
print(" Best Parameters Found:")
print(grid_search.best_params_)
print(f"\nBest Cross-Validation Score: {grid_search.best_score_:.4f}")

# Get best model
best_rf = grid_search.best_estimator_

# Evaluate on test set
y_pred_optimized = best_rf.predict(X_test)
optimized_accuracy = accuracy_score(y_test, y_pred_optimized)

print(f"\n Test Set Performance:")
print(f"Baseline Accuracy: {baseline_accuracy:.4f}")
print(f"Optimized Accuracy: {optimized_accuracy:.4f}")
print(f"Improvement: {optimized_accuracy - baseline_accuracy:.4f}")

In [ ]:
# Basic XGBoost model
xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    eval_metric='logloss'
)

# Train
xgb_model.fit(X_train, y_train)

# Predict and evaluate
y_pred_xgb = xgb_model.predict(X_test)
xgb_accuracy = accuracy_score(y_test, y_pred_xgb)

print(f" XGBoost Accuracy: {xgb_accuracy:.4f} ({xgb_accuracy*100:.2f}%)")
print("\nXGBoost Classification Report:")
print(classification_report(y_test, y_pred_xgb))

In [ ]:
# XGBoost parameter grid (smaller for speed)
xgb_param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1, 0.3],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1.0]
}

# GridSearch for XGBoost
xgb_grid = GridSearchCV(
    estimator=XGBClassifier(random_state=42, eval_metric='logloss'),
    param_grid=xgb_param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

print(" Tuning XGBoost...")
xgb_grid.fit(X_train, y_train)

# Best XGBoost model
best_xgb = xgb_grid.best_estimator_

print("\n Best XGBoost Parameters:")
print(xgb_grid.best_params_)

# Evaluate
y_pred_xgb_opt = best_xgb.predict(X_test)
xgb_opt_accuracy = accuracy_score(y_test, y_pred_xgb_opt)

print(f"\nOptimized XGBoost Accuracy: {xgb_opt_accuracy:.4f} ({xgb_opt_accuracy*100:.2f}%)")

In [ ]:
# Calculate metrics for all models
models_data = [
    ('Baseline Random Forest', y_pred_baseline),
    ('Optimized Random Forest', y_pred_optimized),
    ('Basic XGBoost', y_pred_xgb),
    ('Optimized XGBoost', y_pred_xgb_opt)
]

comparison_data = []

for name, predictions in models_data:
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions)
    recall = recall_score(y_test, predictions)
    f1 = f1_score(y_test, predictions)
    
    comparison_data.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('Accuracy', ascending=False)

print(" Model Comparison:")
print(comparison_df.to_string(index=False))

In [ ]:
# Plot comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metrics_list = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']

for idx, metric in enumerate(metrics_list):
    ax = axes[idx // 2, idx % 2]
    ax.barh(comparison_df['Model'], comparison_df[metric], color=colors[idx % len(colors)])
    ax.set_xlabel(metric)
    ax.set_title(f'{metric} Comparison')
    ax.set_xlim(0.7, 0.9)
    
    # Add value labels
    for i, v in enumerate(comparison_df[metric]):
        ax.text(v + 0.005, i, f'{v:.4f}', va='center')

plt.tight_layout()
plt.show()

# Best model
best_model_name = comparison_df.iloc[0]['Model']
best_accuracy = comparison_df.iloc[0]['Accuracy']
print(f"\n🏆 Best Model: {best_model_name} with {best_accuracy:.4f} ({best_accuracy*100:.2f}%) accuracy")

In [ ]:
# Confusion matrix for best model
best_predictions = y_pred_xgb_opt  # XGBoost optimized usually best
cm = confusion_matrix(y_test, best_predictions)

# Visualize
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Best Model (Optimized XGBoost)')
plt.show()

# Calculate metrics from confusion matrix
tn, fp, fn, tp = cm.ravel()
print(f"True Negatives: {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives: {tp}")
print(f"\nFalse Positive Rate: {fp/(fp+tn):.4f}")
print(f"False Negative Rate: {fn/(fn+tp):.4f}")

In [ ]:
# Get feature importance from best model
importances = pd.DataFrame({
    'feature': X.columns,
    'importance': best_xgb.feature_importances_
}).sort_values('importance', ascending=False)

# Plot top 15 features
plt.figure(figsize=(10, 8))
plt.barh(importances['feature'].head(15), importances['importance'].head(15))
plt.xlabel('Importance')
plt.title('Top 15 Most Important Features (XGBoost)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("🏆 Top 10 Most Important Features:")
print(importances.head(10).to_string(index=False))

In [ ]:
# Save model
model_filename = 'best_churn_model.pkl'
with open(model_filename, 'wb') as file:
    pickle.dump(best_xgb, file)

print(f" Model saved as {model_filename}")

# Verify we can load it
with open(model_filename, 'rb') as file:
    loaded_model = pickle.load(file)

# Test loaded model
test_sample = X_test[:5]
test_predictions = loaded_model.predict(test_sample)
print("\n Test predictions from loaded model:")
print(test_predictions)

In [ ]:
# Create metadata
metadata = {
    'model_type': 'XGBoost',
    'accuracy': xgb_opt_accuracy,
    'precision': precision_score(y_test, y_pred_xgb_opt),
    'recall': recall_score(y_test, y_pred_xgb_opt),
    'f1_score': f1_score(y_test, y_pred_xgb_opt),
    'best_params': xgb_grid.best_params_,
    'features': list(X.columns)
}

# Save metadata
with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=4)

print(" Metadata saved to model_metadata.json")